In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','--force-reinstall','torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)


In [ ]:
from pathlib import Path
import json,subprocess,torch,numpy as np,pandas as pd,matplotlib.pyplot as plt
assert torch.cuda.is_available(), 'GPU_REQUIRED'
repo=Path('/tmp/PCC'); subprocess.run(['git','clone','--quiet','https://github.com/changxinjiresearch/PCC.git',str(repo)],check=True); subprocess.run(['git','-C',str(repo),'checkout','--quiet','8bcbf6f'],check=True)
nb=json.loads((repo/'archive/pcc-experiments-original.ipynb').read_text()); source=''.join(nb['cells'][74]['source'])
start=source.index('class ConvBlock'); end=source.index('# Main 5-fold experiment')
import torch.nn as nn, torch.nn.functional as F
DEVICE='cuda'; BATCH_SIZE=64; PCC_MAX_DELTA_LOGIT=3.0
exec(compile(source[start:end],'archive-cell-74-definition-slice','exec'),globals())
root=Path('/kaggle/input'); metrics_path=next(root.rglob('layer1_FORMAL_v1_case_metrics.csv')); metrics=pd.read_csv(metrics_path)
selected=[('layer1_improvement',metrics.sort_values(['dice_gain','case_id'],ascending=[False,True]).iloc[0]),('layer1_decline',metrics.sort_values(['dice_gain','case_id']).iloc[0])]
out=Path('/kaggle/working/pcc_internal_completion_2026/07_qualitative_panels'); out.mkdir(parents=True,exist_ok=True); rows=[]
for category,row in selected:
 case=row.case_id; fold=int(row['fold']); npz=next(root.rglob(case+'.npz')); data=np.load(npz,allow_pickle=False)['X'].astype(np.float32); x=data[:,0:1]; target=data[:,1:2]
 bck=next(root.rglob(f'baseline_seg_fold_{fold}_FORMAL_v1.pt')); pck=next(root.rglob(f'pcc_seg_corrector_fold_{fold}_FORMAL_v1.pt'))
 bd=torch.load(bck,map_location=DEVICE,weights_only=False); pdict=torch.load(pck,map_location=DEVICE,weights_only=False)
 baseline=SmallUNet2D(1,1,16).to(DEVICE); baseline.load_state_dict(bd['model_state_dict']); bp=predict_prob(baseline,x)
 corrector=SmallUNet2D(2,1,16).to(DEVICE); corrector.load_state_dict(pdict['model_state_dict']); pp=predict_pcc_corrected(corrector,x,bp)
 z=int(np.argmax(target.reshape(len(target),-1).sum(1))); panels=[('Current T1c',x[z,0],'gray'),('Current mask',target[z,0],'gray'),('Baseline probability',bp[z,0],'viridis'),('PCC probability',pp[z,0],'viridis'),('PCC - baseline',pp[z,0]-bp[z,0],'coolwarm'),('Baseline binary',bp[z,0]>=float(bd['threshold']),'gray'),('PCC binary',pp[z,0]>=float(pdict['threshold']),'gray')]
 fig,axes=plt.subplots(2,4,figsize=(14,7))
 for ax in axes.flat: ax.axis('off')
 for ax,(title,array,cmap) in zip(axes.flat,panels): ax.imshow(array,cmap=cmap); ax.set_title(title); ax.axis('off')
 fig.suptitle(f'{category}: {case}; axial index {z}; Layer 1 Formal v1'); fig.tight_layout(); stem=f'{category}__{case}'; fig.savefig(out/(stem+'.png'),dpi=300,bbox_inches='tight'); fig.savefig(out/(stem+'.svg'),bbox_inches='tight'); plt.close(fig); rows.append({'category':category,'case_id':case,'fold':fold,'dice_gain':row.dice_gain,'slice_z':z,'baseline_checkpoint':bck.name,'pcc_checkpoint':pck.name,'protocol':'Layer 1 Formal v1','training_performed':False})
pd.DataFrame(rows).to_csv(out/'LAYER1_QUALITATIVE_PANEL_SOURCE_MAP.csv',index=False); (out/'LAYER1_PANELS_COMPLETE.json').write_text(json.dumps({'status':'COMPLETE','panels':2,'selection':'global max/min Formal v1 dice_gain; case_id tie break','training_performed':False},indent=2)+'\n')
